# 01 · Data Modeling & Quality

**Goal:** turn the 8 raw Olist tables into three clean, correctly-grained analytics tables that every later analysis can trust.

| Table | Grain | Used for |
|---|---|---|
| `order_items` | one row per order line | category & basket analysis, recommender training |
| `orders` | one row per order | every KPI and the A/B test |
| `customers` | one row per real customer (`customer_unique_id`) | retention, CLV, segmentation |

The same model is written in SQL in [`sql/03_analytics_model.sql`](../sql/03_analytics_model.sql) and checked in [`sql/04_data_validation.sql`](../sql/04_data_validation.sql). This notebook is the pandas twin, so the project also runs without a database.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from smart_basket import data_model as dm

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

raw = dm.load_raw()
pd.DataFrame({"rows": {k: len(v) for k, v in raw.items()}})

,rows
customers,99441
orders,99441
order_items,112650
order_payments,103886
order_reviews,99224
products,32951
sellers,3095
category_translation,71


## 1. Understand the keys before joining anything

Three facts about the source decide how the model must be built:

In [2]:
c, rv, p = raw["customers"], raw["order_reviews"], raw["order_payments"]
items_per_order = raw["order_items"].groupby("order_id").size()
payments_per_order = p.groupby("order_id").size()

facts = pd.Series(
    {
        "customer_id values": c["customer_id"].nunique(),
        "real customers (customer_unique_id)": c["customer_unique_id"].nunique(),
        "orders with more than one review": int((rv.groupby("order_id").size() > 1).sum()),
        "orders with more than one item": int((items_per_order > 1).sum()),
        "orders with more than one payment": int((payments_per_order > 1).sum()),
        "orders with no items (never fulfilled)": len(set(raw["orders"]["order_id"]) - set(items_per_order.index)),
    }
)
facts.to_frame("count")

,count
customer_id values,99441
real customers (customer_unique_id),96096
orders with more than one review,547
orders with more than one item,9803
orders with more than one payment,2961
orders with no items (never fulfilled),775


* **`customer_id` is per order, not per person.** Olist issues a new `customer_id` for every order, so repeat behaviour must use `customer_unique_id`. Using `customer_id` makes every customer look like a one-time buyer.
* **Items, payments and reviews are all one-to-many with orders.** Joining two of them directly multiplies rows.
* **775 orders have no items.** They were never fulfilled and carry no revenue, so they are excluded.

## 2. The bug in the original version of this project

The first version joined items, payments and reviews straight onto each other. Here is that join reproduced, compared with the correct approach:

In [3]:
naive = (
    raw["order_items"]
    .merge(raw["orders"], on="order_id")
    .merge(raw["order_payments"], on="order_id", how="left")
    .merge(raw["order_reviews"][["order_id", "review_score"]], on="order_id", how="left")
)
true_payments = p[p["order_id"].isin(raw["order_items"]["order_id"])]["payment_value"].sum()

bug = pd.DataFrame(
    {
        "naive join": [len(naive), naive["payment_value"].sum()],
        "correct": [len(raw["order_items"]), true_payments],
    },
    index=["rows (should equal order lines)", "total payments (R$)"],
)
bug["overstatement"] = (bug["naive join"] / bug["correct"] - 1).map("{:+.1%}".format)
bug[["naive join", "correct"]] = bug[["naive join", "correct"]].map("{:,.0f}".format)
bug

,naive join,correct,overstatement
rows (should equal order lines),"118,310","112,650",+5.0%
total payments (R$),"20,416,843","15,846,280",+28.8%


An order with 3 items and 2 payments becomes 6 rows, and each payment is counted three times. Every revenue KPI built on that table was inflated by almost 30%.

**Fix:** aggregate each child table to *order grain first* (sum items, sum payments, keep the latest review), then join one-to-one.

## 3. Build the analytics tables

In [4]:
tables = dm.build_all()          # also caches them to data/processed/
orders, customers, order_items = tables["orders"], tables["customers"], tables["order_items"]
pd.DataFrame({"rows": {k: len(v) for k, v in tables.items()}})

,rows
order_items,112650
orders,98666
customers,94983


`customers` has 94,983 rows, not 96,096: customers whose only orders were canceled or never fulfilled have no valid orders.

In [5]:
orders.head(3).T

,0,1,2
order_id,2e7a8482f6fb09756ca50c10d7bfc047,e5fa5a7210941f7d56d0208e4e071d35,bfbd0f9bdef84302105ad712db648a6c
customer_unique_id,b7d76e111c89f7ebf14761390f0f7d17,4854e9b3feff728c13ee5fc7d1547e92,830d5b7aaa3b6f1e9ad63703bec97d23
customer_state,RR,RS,SP
order_status,shipped,canceled,delivered
purchased_at,2016-09-04 21:15:19,2016-09-05 00:15:34,2016-09-15 12:16:38
purchase_month,2016-09-01 00:00:00,2016-09-01 00:00:00,2016-09-01 00:00:00
is_delivered,False,False,True
is_canceled,False,True,False
items_count,2,1,3
distinct_products,2,1,1


## 4. Validate: reconcile, check grain, check logic

Mirrors `sql/04_data_validation.sql`. If any check fails, the notebook stops here.

In [6]:
paid = orders["payment_value"].notna()
checks = {
    "one row per order": orders["order_id"].is_unique,
    "every order with items is present": len(orders) == raw["order_items"]["order_id"].nunique(),
    "item revenue reconciles to raw (to the cent)": abs(orders["item_revenue"].sum() - raw["order_items"]["price"].sum()) < 0.01,
    "payments reconcile to raw (to the cent)": abs(orders["payment_value"].sum() - true_payments) < 0.01,
    "payment ≈ price + freight on >95% of orders": ((orders.loc[paid, "payment_value"] - orders.loc[paid, "gmv"]).abs() <= 1).mean() > 0.95,
    "each customer has exactly one first order": (orders.groupby("customer_unique_id")["customer_order_number"].min() == 1).all()
    and (orders["customer_order_number"] == 1).sum() == orders["customer_unique_id"].nunique(),
    "no delivery before purchase": (orders["delivery_days"].dropna() >= 0).all(),
    "review scores within 1–5": orders["review_score"].dropna().between(1, 5).all(),
    "customer table reconciles with valid orders": customers["valid_orders"].sum() == (~orders["is_canceled"]).sum(),
}
result = pd.Series(checks).map({True: "PASS", False: "FAIL"}).to_frame("status")
assert (result["status"] == "PASS").all(), result
result

,status
one row per order,PASS
every order with items is present,PASS
item revenue reconciles to raw (to the cent),PASS
payments reconcile to raw (to the cent),PASS
payment ≈ price + freight on >95% of orders,PASS
each customer has exactly one first order,PASS
no delivery before purchase,PASS
review scores within 1–5,PASS
customer table reconciles with valid orders,PASS


## 5. Data-quality profile

In [7]:
profile = orders["order_status"].value_counts().to_frame("orders")
profile["share"] = (profile["orders"] / profile["orders"].sum()).map("{:.2%}".format)
profile

,orders,share
order_status,,
delivered,96478,97.78%
shipped,1106,1.12%
canceled,461,0.47%
invoiced,312,0.32%
processing,301,0.31%
unavailable,6,0.01%
approved,2,0.00%


In [8]:
pd.Series(
    {
        "first purchase": orders["purchased_at"].min(),
        "last purchase": orders["purchased_at"].max(),
        "orders without a review": int(orders["review_score"].isna().sum()),
        "orders not delivered (yet)": int(orders["delivery_days"].isna().sum()),
        "order lines with unknown category": f"{(order_items['category'] == 'unknown').mean():.2%}",
        "real repeat-customer rate": f"{customers['is_repeat_customer'].mean():.2%}",
    }
).to_frame("value")

,value
first purchase,2016-09-04 21:15:19
last purchase,2018-09-03 09:06:57
orders without a review,749
orders not delivered (yet),2190
order lines with unknown category,1.42%
real repeat-customer rate,3.04%


### Modeling decisions

* **Canceled & unavailable orders** (0.5%) stay in `orders` with an `is_canceled` flag. Revenue KPIs exclude them; the A/B test keeps them, because cancellation rate is a guardrail metric.
* **Trend window:** Sep–Dec 2016 has only a few hundred orders, and Sep 2018 is a partial month. Trend charts use **Jan 2017 – Aug 2018**.
* **GMV = item price + freight.** This is what the customer pays before installment interest, and it reconciles with payments on 99.7% of orders.
* **Repeat customer** = 2+ non-canceled orders under the same `customer_unique_id`. The true rate is **3.0%**; the original version reported 19% because it flagged rows rather than customers.
* **Geolocation** (1M rows, many duplicate points per zip code) isn't needed for these questions and is not loaded.